# GeoZones — Global zones

Downloads the geozones archive from data.gouv.fr and extracts `zones.json` (655 MB),
then pulls out the 6 top-level geographic containers (world, EU, fr, fr:metro, fr:drom, fr:dromcom).

Output: `zones_global.geojson`

In [4]:
import subprocess
from collections import defaultdict

try:
    import ijson
except ImportError:
    subprocess.check_call(["uv", "add", "ijson"])
    import ijson

DOWNLOAD_URL = "https://www.data.gouv.fr/api/1/datasets/r/4ec9e77d-572a-40f9-9940-ade023ce8b78"
TAR_FILE     = "./geozones.tar.xz"
ZONES_FILE   = "./zones.json"
GLOBAL_FILE  = "./zones_global.geojson"

print("ijson", ijson.__version__)

ijson 3.5.0


In [5]:
import tarfile, urllib.request, os

if not os.path.exists(TAR_FILE):
    print(f"Downloading {DOWNLOAD_URL} ...")
    urllib.request.urlretrieve(DOWNLOAD_URL, TAR_FILE)
    print(f"Downloaded. ({os.path.getsize(TAR_FILE)/1024/1024:.0f} MB)")
else:
    print(f"Archive present: {TAR_FILE}  ({os.path.getsize(TAR_FILE)/1024/1024:.0f} MB)")

if not os.path.exists(ZONES_FILE):
    print("Extracting zones.json ...")
    with tarfile.open(TAR_FILE, "r:xz") as tar:
        tar.extract("zones.json", ".")
    print(f"Done. ({os.path.getsize(ZONES_FILE)/1024/1024:.0f} MB)")
else:
    print(f"Already extracted: {ZONES_FILE}  ({os.path.getsize(ZONES_FILE)/1024/1024:.0f} MB)")

Downloaded. (57 MB)
Extracting zones.json ...


Done. (655 MB)


In [6]:
import json, os

TOP_IDS = {
    "country-group:world",
    "country-group:ue",
    "country:fr",
    "country-subset:fr:metro",
    "country-subset:fr:drom",
    "country-subset:fr:dromcom",
}

first = True
count = 0
with open(ZONES_FILE, "rb") as src, open(GLOBAL_FILE, "w") as dst:
    dst.write('{"type":"FeatureCollection","features":[')
    for feature in ijson.items(src, "features.item", use_float=True):
        fid = feature.get("id", "").split("@")[0]
        if fid in TOP_IDS:
            if not first:
                dst.write(",")
            feature["id"] = fid
            dst.write(json.dumps(feature, ensure_ascii=False))
            first = False
            count += 1
    dst.write("]}")

print(f"Done. {count} global zones → {GLOBAL_FILE}  ({os.path.getsize(GLOBAL_FILE)/1024:.1f} KB)")

Done. 6 global zones → ./zones_global.geojson  (1161.8 KB)
